# 06. Futures Wheel — 練習問題

**対象技術**: AI（生成AI・機械学習・自動化を含む）

Futures Wheel は、中心命題から一次〜三次の波及効果を放射状に展開する手法である。本ノートブックでは、波及効果をネストした木構造で表現し、各枝に条件付き発生確率とインパクト値を付与して、根からの累積確率と期待インパクトを再帰的に伝播計算する。深さ別の効果数集計とホットスポット抽出を行い、最後に波及効果の放射図を可視化する。

必要なライブラリを読み込む。

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

# --- 日本語フォント設定: 共通モジュール jp_font.py を読み込む ---
# フォント探索・登録・フォールバックの実装は repo 直下の jp_font.py に集約。
import os as _os, sys as _sys
_d = _os.path.abspath(_os.getcwd())
while not _os.path.exists(_os.path.join(_d, "jp_font.py")) and _d != _os.path.dirname(_d):
    _d = _os.path.dirname(_d)
_sys.path.insert(0, _d)
from jp_font import setup_japanese_font
setup_japanese_font()


波及効果の1ノードを表すクラスを定義する。子ノードを再帰的に保持する木構造になっている。

In [ ]:
class EffectNode:
    """波及効果の1ノード。子ノードを再帰的に保持する木構造。"""

    def __init__(self, label, cond_prob, impact, steep, children=None):
        # cond_prob : 親が起きたとき、この効果が起きる条件付き確率
        # impact    : この効果固有のインパクト規模（1=軽微 〜 10=甚大）
        # steep     : STEEP 分類タグ（S/T/E/En/P）
        self.label = label
        self.cond_prob = cond_prob
        self.impact = impact
        self.steep = steep
        self.children = children if children is not None else []

AI を題材にした Futures Wheel を構築する。中心命題「汎用的な生成AIの社会全体への普及」から一次・二次・三次効果を木構造で組み立てる。

In [ ]:
def build_ai_wheel():
    """生成AIの社会普及を題材にした Futures Wheel を構築する。"""
    # --- 三次効果 ---
    job_restructure = EffectNode("雇用構造の再編", 0.65, 9, "E")
    media_rebuild = EffectNode("メディアエコシステムの再構築", 0.55, 8, "S")
    # --- 二次効果（一次:知識労働の自動化 から）---
    job_decline = EffectNode("一部職種の需要減", 0.70, 7, "E",
                             [job_restructure])
    new_jobs = EffectNode("新職種の発生", 0.75, 6, "E")
    skill_shift = EffectNode("スキル要件の変化", 0.90, 6, "S")
    # --- 二次効果（一次:偽情報の氾濫 から）---
    trust_decline = EffectNode("情報の信頼性低下", 0.85, 8, "S",
                               [media_rebuild])
    verify_cost = EffectNode("検証コストの増大", 0.80, 6, "E")
    platform_reg = EffectNode("プラットフォーム規制圧力", 0.75, 7, "P")
    # --- 一次効果 ---
    knowledge_auto = EffectNode("知識労働の部分自動化", 0.85, 8, "E",
                                [job_decline, new_jobs, skill_shift])
    content_cost = EffectNode("コンテンツ生成コストの急落", 0.90, 6, "E")
    education = EffectNode("教育・学習の変容", 0.70, 6, "S")
    disinfo = EffectNode("偽情報・ディープフェイクの氾濫", 0.75, 8, "S",
                         [trust_decline, verify_cost, platform_reg])
    # --- 中心命題（根。条件付き確率は前提なので 1.0）---
    root = EffectNode("汎用的な生成AIの社会全体への普及", 1.0, 0, "T",
                      [knowledge_auto, content_cost, education,
                       disinfo])
    return root

根からの累積確率（条件付き確率の積）と期待インパクト（累積確率 × インパクト値）を再帰で伝播計算する関数を定義する。

In [ ]:
def propagate(node, parent_cum_prob, depth, records):
    """根からの累積確率と期待インパクトを再帰で伝播計算する。"""
    cum_prob = parent_cum_prob * node.cond_prob          # 累積確率 = 条件付き確率の積
    expected_impact = cum_prob * node.impact             # 期待インパクト
    if depth >= 1:  # 根（中心命題）自体は集計対象外
        records.append({
            "label": node.label, "depth": depth, "steep": node.steep,
            "cum_prob": cum_prob, "impact": node.impact,
            "expected_impact": expected_impact,
        })
    for child in node.children:
        propagate(child, cum_prob, depth + 1, records)

Futures Wheel を構築し、累積確率・期待インパクトを伝播させて、深さ順に波及効果の一覧を表示する。

In [ ]:
np.random.seed(0)
root = build_ai_wheel()

# 累積確率・期待インパクトを伝播
records = []
propagate(root, 1.0, 0, records)

print("=" * 68)
print("Futures Wheel : 生成AI社会普及の波及効果（累積確率の伝播）")
print("=" * 68)
print(f"{'深さ':<4}{'効果':<34}{'累積確率':>9}{'期待ｲﾝﾊﾟｸﾄ':>11}")
print("-" * 68)
for r in sorted(records, key=lambda x: (x["depth"], -x["expected_impact"])):
    indent = "  " * (r["depth"] - 1)
    name = (indent + r["label"])[:33]
    print(f"{r['depth']:<4}{name:<34}{r['cum_prob']:>9.3f}"
          f"{r['expected_impact']:>11.2f}")

深さ別の効果数を集計する。

In [ ]:
print("--- 深さ別の効果数 ---")
for d in sorted(set(r["depth"] for r in records)):
    n = sum(1 for r in records if r["depth"] == d)
    order = {1: "一次", 2: "二次", 3: "三次"}.get(d, f"{d}次")
    print(f"  {order}効果 : {n} 件")

期待インパクト上位3件をホットスポットとして抽出する。「起こりやすく、かつ影響が大きい」効果が重点対象となる。

In [ ]:
print("--- ホットスポット（期待インパクト上位3件）---")
top = sorted(records, key=lambda x: -x["expected_impact"])[:3]
for i, r in enumerate(top, 1):
    print(f"  {i}. {r['label']}")
    print(f"     累積確率 {r['cum_prob']:.2f} × インパクト {r['impact']} "
          f"= 期待値 {r['expected_impact']:.2f}")

print()
print("[解釈] 二次効果『情報の信頼性低下』は累積確率・影響規模")
print("       がともに高くホットスポットに入る。対策の起点は")
print("       モデル側の検出技術だけでなく、情報流通の")
print("       検証インフラと制度設計に置く必要がある。")

## 可視化: 波及効果の放射図

中心ノードから一次→二次→三次効果を同心円上に配置し、枝を線で結ぶ。各ノードの大きさと色を期待インパクトに対応させ、ホットスポット上位3件を強調表示する。ラベルは英数字 ID（深さ-連番）で表す。

In [ ]:
import math

# 木構造を (親ID, 子ノード) のリストとして展開し、各ノードにIDを振る
node_info = {}   # id -> dict(label, depth, expected_impact, cum_prob)
edges = []       # (parent_id, child_id)

def assign_ids(node, depth, parent_id, counters):
    if depth == 0:
        nid = "C"
    else:
        counters[depth] = counters.get(depth, 0) + 1
        nid = f"{depth}-{counters[depth]}"
    cum = (node_info[parent_id]["cum_prob"] if parent_id else 1.0) * node.cond_prob
    node_info[nid] = {"label": node.label, "depth": depth,
                      "cum_prob": cum, "impact": node.impact,
                      "expected_impact": cum * node.impact}
    if parent_id is not None:
        edges.append((parent_id, nid))
    for child in node.children:
        assign_ids(child, depth + 1, nid, counters)

assign_ids(root, 0, None, {})

# 同心円上に配置: 深さ d は半径 d、同じ深さのノードを角度で等分
pos = {}
by_depth = {}
for nid, info in node_info.items():
    by_depth.setdefault(info["depth"], []).append(nid)
for d, ids in by_depth.items():
    ids_sorted = sorted(ids)
    for k, nid in enumerate(ids_sorted):
        if d == 0:
            pos[nid] = (0.0, 0.0)
        else:
            ang = 2 * math.pi * k / len(ids_sorted) + 0.3 * d
            pos[nid] = (d * math.cos(ang), d * math.sin(ang))

# ホットスポット上位3件のID
hot = set(nid for nid, _ in sorted(
    [(nid, info["expected_impact"]) for nid, info in node_info.items()
     if info["depth"] >= 1], key=lambda x: -x[1])[:3])

fig, ax = plt.subplots(figsize=(9, 9))
# 枝（線）
for p, c in edges:
    x = [pos[p][0], pos[c][0]]
    y = [pos[p][1], pos[c][1]]
    ax.plot(x, y, color="gray", lw=1.0, zorder=1)
# ノード
for nid, info in node_info.items():
    x, y = pos[nid]
    ei = info["expected_impact"]
    size = 300 + ei * 220
    if nid == "C":
        ax.scatter([x], [y], s=900, color="black", zorder=3)
        ax.annotate("CENTER", (x, y), ha="center", va="center",
                    color="white", fontsize=8, zorder=4)
    else:
        color = plt.cm.YlOrRd(min(ei / 6.0, 1.0))
        edge = "blue" if nid in hot else "none"
        lw = 2.5 if nid in hot else 0.0
        ax.scatter([x], [y], s=size, color=color, edgecolors=edge,
                   linewidths=lw, zorder=3)
        ax.annotate(nid, (x, y), ha="center", va="center",
                    fontsize=8, zorder=4)

ax.set_title("Futures Wheel: ripple effects (size/color = expected impact, "
             "blue ring = hotspot)")
ax.set_aspect("equal")
ax.axis("off")

# 凡例代わりにIDとラベルの対応表をテキストで表示
legend_lines = [f"{nid}: {info['label']} (EI={info['expected_impact']:.2f})"
                for nid, info in sorted(node_info.items())
                if info["depth"] >= 1]
ax.text(1.05, 1.0, "\n".join(legend_lines), transform=ax.transAxes,
        va="top", ha="left", fontsize=7, family="monospace")
plt.show()

## 未来デザイン論文での使われ方と結論への影響

未来デザインの論文において Futures Wheel は、ある技術や政策を中心命題に据え、そこから一次・二次・三次の波及効果を放射状に展開するために用いられる。論文の典型的な使われ方は、当初の政策議題には上っていなかった間接的な効果——制度や規範、別の産業への玉突き——を明示的に可視化し、「これまでの議論は争点の取り方が狭すぎた」と主張するための土台を提供することである。したがってこの手法が生み出す結論は、確定的な予測ではなく争点そのものの組み替えであり、発散的な影響マップという型をとる。

結論の型は「最も重大なのは、これまで見落とされていた二次・三次効果Xである」という形に収束しやすい。これは三つの経路を通じて規定される。境界設定の面では、中心命題と最初に書き出した一次効果が放射の起点を決めるため、起点に置かれなかった主題は枝として伸びず結論に反映されない。時間観の面では、未来を波及の連鎖として描くため、未来は一点に確定するのではなく分岐し広がるものとして扱われる。価値の所在は、どの枝を「重大」と評価しホットスポットに選ぶかという段階に埋め込まれ、その判断は分析者の関心を色濃く反映する。

限界として、Futures Wheel は議論の幅を生むことには長けるが、各枝の発生確率を厳密に評価する力や、効果どうしが互いを強め合う・打ち消し合うフィードバックを表現する力に乏しい。木構造は枝の独立を暗黙に仮定するため、現実の相互依存が結論から抜け落ちる。結果として、この手法に依拠した論文の結論は「考慮すべき争点が増えた」ことは強く示せるが、「そのうちどれが実際に起きるか」には答えられない。幅の広さと確からしさのトレードオフを自覚して読む必要がある。

## 発展課題

**課題A**: 各ノードに STEEP 分類（社会/技術/経済/環境/政治）を属性として追加し、分類別に「効果数」と「期待インパクト合計」を集計せよ。どの次元の枝が薄いか（網羅性の欠落）を点検できるようにする。
**課題B**: 一次効果「偽情報・ディープフェイクの氾濫」の条件付き確率を 0.5〜1.0 で変化させ、三次効果「メディアエコシステムの再構築」の期待インパクトがどう動くかを表にせよ（感度分析）。確率の主観性が結論に与える影響を観察する。